# geoseg walkthrough

A short, GPU-free tour of the tested pure-numpy core: the metric demo, per-class IoU, the multi-class confusion matrix, and a tiling round-trip. Everything here runs on numpy alone (no torch, no GPU, no real data), so it reproduces anywhere the test suite runs. The numbers are seeded and deterministic; run the cells top to bottom.

## 1. Run the demo

`run_demo(0)` synthesises a small 4-class label map and a noisy prediction (15% of pixels flipped to a wrong class), drives the real metric functions over them, and writes `outputs/{per_class_iou.csv, confusion.csv, summary.json}`.

In [ ]:
from geoseg.demo import run_demo

result = run_demo(seed=0, out_dir="outputs")
result

## 2. Per-class IoU

Background (class 0) is large and easy, so its IoU is highest; the smaller blob classes are noisier. `mean_iou_multiclass` macro-averages the classes that are present.

In [ ]:
for k, iou in enumerate(result["per_class_iou"]):
    print(f"class {k}: IoU = {iou:.4f}")
print(f"\nmean IoU = {result['mean_iou']:.4f}")
print(f"pixel accuracy = {result['pixel_accuracy']:.4f}")

## 3. Confusion matrix

Rows are the true class, columns the predicted class; the diagonal holds the correctly labelled pixels. Off-diagonal mass shows which classes get confused for which. We rebuild the same seeded pair the demo used so the matrix matches the summary above.

In [ ]:
import numpy as np
from geoseg.metrics import confusion_matrix, cohen_kappa, frequency_weighted_iou
from geoseg.demo import _synthesize

rng = np.random.default_rng(0)
truth, pred = _synthesize(rng, 64, 64, 4, 0.15)
cm = confusion_matrix(pred, truth, num_classes=4)
print(cm)
print(f"\ncohen kappa = {cohen_kappa(pred, truth, 4):.4f}")
print(f"frequency-weighted IoU = {frequency_weighted_iou(pred, truth, 4):.4f}")

## 4. Tiling round-trip

`tile_indices` enumerates sliding windows over an image and `stitch` reassembles per-tile arrays, averaging overlaps. With overlapping windows over identical source pixels the reconstruction is exact.

In [ ]:
from geoseg.tiling import tile_indices, stitch

img = rng.random((10, 10))
windows = tile_indices(10, 10, tile=4, overlap=2)
tiles = [img[r0:r1, c0:c1] for (r0, r1, c0, c1) in windows]
recon = stitch(tiles, windows, 10, 10)
print(f"{len(windows)} windows cover the 10x10 image")
print(f"max abs reconstruction error = {np.abs(recon - img).max():.2e}")
assert np.allclose(recon, img)